In [ ]:
pip install hmmlearn scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

# Disable requests_cache if active
try:
    import requests_cache
    requests_cache.uninstall_cache()
except ImportError:
    pass

session = requests.Session()

# =====================================================================
# 1. FETCH EURO 2024 FINAL (SPAIN VS ENGLAND)
# =====================================================================
print("Fetching Euro 2024 Final...")
# Match ID for the Euro 2024 Final is 3943043
events_url = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3943043.json"
events = session.get(events_url).json()
print(f"✅ Loaded {len(events)} events from the Final.")

# =====================================================================
# 2. FEATURE ENGINEERING: 5-MINUTE ROLLING METRICS
# =====================================================================
target_team = "England"
bin_size = 5 # 5-minute windows

# We will collect data for every 5 minute bin
bins = {}
max_minute = max(e.get('minute', 0) for e in events)
num_bins = (max_minute // bin_size) + 1

for i in range(num_bins):
    bins[i] = {
        'eng_passes': 0, 'eng_forward_passes': 0, 'total_passes': 0,
        'def_x_coords': []
    }

defensive_actions = ['Tackle', 'Interception', 'Clearance', 'Block', 'Ball Recovery']

for event in events:
    if 'minute' not in event: continue
    bin_idx = event['minute'] // bin_size
    team_name = event.get('team', {}).get('name')
    type_name = event.get('type', {}).get('name')
    
    # Passing metrics (for Possession and Verticality)
    if type_name == 'Pass':
        bins[bin_idx]['total_passes'] += 1
        if team_name == target_team:
            bins[bin_idx]['eng_passes'] += 1
            # Check if pass went forward (End X > Start X)
            loc = event.get('location', [0, 0])
            end_loc = event.get('pass', {}).get('end_location', [0, 0])
            if end_loc[0] > loc[0]:
                bins[bin_idx]['eng_forward_passes'] += 1
                
    # Defensive Action Metrics (for Line Height)
    if type_name in defensive_actions and team_name == target_team:
        loc = event.get('location')
        if loc:
            bins[bin_idx]['def_x_coords'].append(loc[0])

# Compile into a DataFrame
data = []
for i in range(num_bins):
    eng_p = bins[i]['eng_passes']
    tot_p = bins[i]['total_passes']
    fwd_p = bins[i]['eng_forward_passes']
    def_coords = bins[i]['def_x_coords']
    
    # Calculate Metrics
    possession = (eng_p / tot_p * 100) if tot_p > 0 else 0
    verticality = (fwd_p / eng_p * 100) if eng_p > 0 else 0
    def_line = np.mean(def_coords) if len(def_coords) > 0 else np.nan
    
    data.append({
        'minute_start': i * bin_size,
        'possession_pct': possession,
        'verticality_pct': verticality,
        'defensive_line_x': def_line
    })

df = pd.DataFrame(data)

# Forward fill missing defensive lines (if a team had no def actions in 5 mins, assume they held their last line)
df['defensive_line_x'] = df['defensive_line_x'].ffill().bfill() 

print("\n✅ Engineered 5-minute tactical windows.")

# =====================================================================
# 3. TRAIN THE HIDDEN MARKOV MODEL (HMM)
# =====================================================================
# Features to feed the AI
features = ['possession_pct', 'verticality_pct', 'defensive_line_x']
X_raw = df[features].values

# Standardize the data so the HMM treats all metrics equally
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# We ask the AI to find exactly 3 Hidden Tactical States
print("Training Hidden Markov Model...")
model = hmm.GaussianHMM(n_components=3, covariance_type="diag", n_iter=1000, random_state=42)
model.fit(X_scaled)

# Predict the hidden state for every 5-minute window
hidden_states = model.predict(X_scaled)
df['hidden_state'] = hidden_states

# =====================================================================
# 4. ANALYZE THE STATES (PRINT TO TERMINAL)
# =====================================================================
print("\n=== AI DISCOVERED TACTICAL STATES (ENGLAND) ===")
state_profiles = df.groupby('hidden_state')[features].mean()

for state in range(3):
    profile = state_profiles.loc[state]
    count = (df['hidden_state'] == state).sum()
    print(f"\nState {state} (Occurred for {count * 5} minutes):")
    print(f"  - Possession: {profile['possession_pct']:.1f}%")
    print(f"  - Forward Pass %: {profile['verticality_pct']:.1f}%")
    print(f"  - Avg Defensive Line (X-Coord): {profile['defensive_line_x']:.1f} yards")

# =====================================================================
# 5. PLOT GRAPH 1: THE TACTICAL TIMELINE
# =====================================================================
fig, ax = plt.subplots(figsize=(14, 6), facecolor='white')

# Plot a metric (e.g., Defensive Line) over time
ax.plot(df['minute_start'], df['defensive_line_x'], color='black', linewidth=2, marker='o', label="Defensive Line Height")

# Shade the background based on the Hidden State
colors = {0: '#ff9999', 1: '#99ccff', 2: '#99ff99'}
for i in range(len(df)):
    ax.axvspan(df['minute_start'].iloc[i], df['minute_start'].iloc[i] + bin_size, 
               color=colors[df['hidden_state'].iloc[i]], alpha=0.4)

# Custom legend for the states
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=colors[i], alpha=0.4, label=f'Hidden State {i}') for i in range(3)]
ax.legend(handles=patches + [ax.lines[0]], loc='upper right')

plt.title("England's Hidden Tactical States (Euro 2024 Final)", fontsize=16, fontweight='bold')
plt.xlabel("Match Minute", fontsize=12)
plt.ylabel("Defensive Line Height (Yards)", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('graph1_HS.png', dpi=150, bbox_inches='tight')
plt.show()

# =====================================================================
# 6. PLOT GRAPH 2: STATE PROFILES (BAR CHART)
# =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='white')
state_profiles.reset_index(inplace=True)

sns.barplot(x='hidden_state', y='possession_pct', data=state_profiles, ax=axes[0], palette=colors.values())
axes[0].set_title('Ball Possession %')

sns.barplot(x='hidden_state', y='verticality_pct', data=state_profiles, ax=axes[1], palette=colors.values())
axes[1].set_title('Forward Pass % (Verticality)')

sns.barplot(x='hidden_state', y='defensive_line_x', data=state_profiles, ax=axes[2], palette=colors.values())
axes[2].set_title('Defensive Line Height (Yards)')

plt.suptitle("The DNA of the Hidden States", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('graph2_DNA.png', dpi=150, bbox_inches='tight')
plt.show()